In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any


@dataclass
class QuizJob:
    """
    Represents a submitted answer.
    """

    user_id: str
    project_id: str
    assessment_id: str
    question_id: str
    answer: str

    created_at: datetime = field(
        default_factory=lambda:
            datetime.now(timezone.utc)
    )


class QuizWorker:
    """
    Handles post-answer processing.

    Pipeline:

        submitted answer
               ↓
        answer evaluation
               ↓
        mastery signal
               ↓
        mastery update
               ↓
        recommendation refresh
    """

    def __init__(
        self,
        assessment_service=None,
        mastery_service=None,
        recommendation_service=None,
        database=None,
    ):

        self.assessment_service = (
            assessment_service
        )

        self.mastery_service = (
            mastery_service
        )

        self.recommendation_service = (
            recommendation_service
        )

        self.database = database

    # ========================================================
    # NORMALIZE EVALUATION
    # ========================================================

    @staticmethod
    def _normalize_evaluation(
        evaluation,
    ) -> dict[str, Any]:

        if isinstance(
            evaluation,
            dict,
        ):

            return {
                "is_correct": bool(
                    evaluation.get(
                        "is_correct",
                        False,
                    )
                ),
                "score": float(
                    evaluation.get(
                        "score",
                        0.0,
                    )
                ),
                "feedback": str(
                    evaluation.get(
                        "feedback",
                        "",
                    )
                ),
                "missing_concepts": list(
                    evaluation.get(
                        "missing_concepts",
                        [],
                    )
                ),
                "misconceptions": list(
                    evaluation.get(
                        "misconceptions",
                        [],
                    )
                ),
            }

        # Supports AnswerEvaluator's EvaluationResult
        # dataclass without forcing the evaluator itself
        # to change its public API.
        return {
            "is_correct": bool(
                getattr(
                    evaluation,
                    "is_correct",
                    False,
                )
            ),
            "score": float(
                getattr(
                    evaluation,
                    "score",
                    0.0,
                )
            ),
            "feedback": str(
                getattr(
                    evaluation,
                    "feedback",
                    "",
                )
            ),
            "missing_concepts": list(
                getattr(
                    evaluation,
                    "missing_concepts",
                    [],
                )
            ),
            "misconceptions": list(
                getattr(
                    evaluation,
                    "misconceptions",
                    [],
                )
            ),
        }

    # ========================================================
    # PROCESS
    # ========================================================

    async def process(
        self,
        job: QuizJob,
        question: dict[str, Any],
    ) -> dict[str, Any]:

        if self.assessment_service is None:
            raise RuntimeError(
                "AssessmentService has not been configured."
            )

        question_type = question.get(
            "type",
            "mcq",
        )

        # ====================================================
        # MCQ
        # ====================================================

        if question_type == "mcq":

            correct_option = question.get(
                "correct_option"
            )

            submitted_answer = (
                job.answer.strip()
            )

            is_correct = (
                correct_option is not None
                and submitted_answer
                == str(
                    correct_option
                ).strip()
            )

            evaluation = {
                "is_correct": is_correct,
                "score": (
                    1.0
                    if is_correct
                    else 0.0
                ),
                "feedback": (
                    "Correct."
                    if is_correct
                    else "Incorrect."
                ),
                "missing_concepts": (
                    []
                    if is_correct
                    else list(
                        question.get(
                            "expected_concepts",
                            [],
                        )
                    )
                ),
                "misconceptions": [],
            }

        # ====================================================
        # OPEN ENDED
        # ====================================================

        elif question_type == "open_ended":

            evaluation = (
                self.assessment_service
                .evaluate_open_ended(
                    project_id=job.project_id,
                    user_id=job.user_id,
                    question=question.get(
                        "question",
                        "",
                    ),
                    answer=job.answer,
                    expected_concepts=question.get(
                        "expected_concepts",
                        [],
                    ),
                    source_chunk_ids=question.get(
                        "source_chunk_ids",
                        [],
                    ),
                )
            )

        else:

            raise ValueError(
                f"Unsupported question type: "
                f"{question_type}"
            )

        # ====================================================
        # NORMALIZE
        # ====================================================

        evaluation = (
            self._normalize_evaluation(
                evaluation
            )
        )

        evaluation["score"] = max(
            0.0,
            min(
                1.0,
                float(
                    evaluation["score"]
                ),
            ),
        )

        # ====================================================
        # MASTERY
        # ====================================================

        mastery = None

        if self.mastery_service is not None:

            concept_id = question.get(
                "concept_id"
            )

            if concept_id:

                mastery = (
                    self.mastery_service
                    .update_from_evidence(
                        user_id=job.user_id,
                        project_id=job.project_id,
                        concept_id=concept_id,
                        evidence_score=(
                            evaluation["score"]
                        ),
                        evidence_type=(
                            "quiz_answer"
                        ),
                        evidence_description=(
                            evaluation["feedback"]
                        ),
                    )
                )

        # ====================================================
        # RECOMMENDATIONS
        # ====================================================

        recommendations = []

        if (
            self.recommendation_service
            is not None
            and mastery is not None
        ):

            try:

                recommendations = (
                    self.recommendation_service
                    .generate_for_project(
                        user_id=job.user_id,
                        project_id=job.project_id,
                    )
                )

            except Exception:
                recommendations = []

        return {
            "assessment_id": job.assessment_id,
            "question_id": job.question_id,
            "evaluation": evaluation,
            "mastery": mastery,
            "recommendations": recommendations,
            "status": "completed",
        }
